In [2]:
from dotenv import load_dotenv
from pathlib import Path
from collections import Counter
import os, json
import re
import time
from qdrant_client import QdrantClient, models
from qdrant_client.migrate import migrate
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
import wikipediaapi
import tiktoken
from pypdf import PdfReader
import urllib.parse
import hashlib

from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from typing import List, Optional
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langfuse.langchain import CallbackHandler
from langfuse import get_client

import voyageai
from anthropic import Anthropic

load_dotenv()

True

In [3]:
# Configurations

anthropic_client = Anthropic()

cloud_client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    timeout=120,
)

voyageai_client = voyageai.Client(
    api_key=os.getenv("VOYAGEAI_API_KEY")
)

wiki_client = wikipediaapi.Wikipedia(
    user_agent="MyScraper/1.0",
    language="en",
    extract_format=wikipediaapi.ExtractFormat.WIKI,
)

voyage_model = "voyage-4"
collection_name = "motorsports_corpus"
embed_dim = 1024

In [4]:
langfuse_handler = CallbackHandler()
langfuse = get_client()

In [ ]:
anthropic_model = os.getenv("ANTHROPIC_MODEL")


In [85]:
# Scraping Wikipedia Links

text_content = []

wiki_urls = Path(r"location_to_doc")
DROP_SECTIONS = {"See also", "Reference", "External Links", "Notes", "Further reading"}

def extract_title_from_url(url):
    match = re.search(r"wiki/([^#\?]+)", url)
    if match:
        return urllib.parse.unquote(match.group(1)).replace("_", " ")
    return None

def page_to_markdown(page):
    md = [f"# {page.title}\n\n{page.summary.strip()}\n"]

    def walk(sections, level):
        for s in sections:
            if s.title in DROP_SECTIONS:
                continue

            if s.text.strip():
                md.append(f"{'#' * level} {s.title}\n\n{s.text.strip()}\n")
            walk(s.sections, level + 1)
    walk(page.sections, 2)
    return "\n".join(md)

def wiki_extraction():
    section = None

    with open(wiki_urls, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("#"):
                section = line.lstrip("#").strip()
                continue

            url = line
            clean_title = extract_title_from_url(url)

            try:
                page = wiki_client.page(clean_title)

                if page.exists():
                    text_content.append({
                        "text": page_to_markdown(page),
                        "source": url,
                        "section": section,
                        "doc_type": "wikipedia",
                    }
                )

            except Exception as e:
                print(f"Failed scraping {clean_title} from [{url}]: {e}")

            time.sleep(0.5)
    print(f"{len(text_content)} wikipedias scraped")

In [86]:
wiki_extraction()

41 wikipedias scraped


In [ ]:
# Loading PDFs

pdf_path = Path(r"location_to_pdfs")
pdf_search = pdf_path.rglob("*.pdf")
pdf_files = list(pdf_search)

for pdf in pdf_files:
    reader = PdfReader(pdf)
    pdf_text = ""

    for page in reader.pages:
        text = page.extract_text()
        if text:
            pdf_text += text + "\n"

    if pdf_text.strip():
        text_content.append({
            "text": pdf_text,
            "source": pdf.name,
            "section": pdf.stem,
            "doc_type": "pdf",
        })

print(f"{len(pdf_files)} PDFs loaded")

In [ ]:
# Tokenizing and Chunking

enc = tiktoken.get_encoding("cl100k_base")

def token_len(text):
    return len(enc.encode(text))

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")]
)

wiki_token_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    length_function=token_len,
)

# Wikipedia chunking

def chunk_article(md_text):
    header_chunks = header_splitter.split_text(md_text)
    final_chunks = []
    for hc in header_chunks:
        if token_len(hc.page_content) <= 80:
            final_chunks.append(hc.page_content)
        else:
            final_chunks.extend(wiki_token_splitter.split_text(hc.page_content))
    return final_chunks

# PDFs chunking

pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=280,
    chunk_overlap=40,
    length_function=token_len,
)

def strip_boilerplate(text, min_repeat=5, max_len=90):
    lines = text.split("\n")
    counts = Counter(l.strip() for l in lines if l.strip())
    boilerplate = {l for l, c in counts.items() if c >= min_repeat and len(l) <= max_len}
    page_marker = re.compile(r"^[A-Z]{1,2}\d{1,4}$")
    cleaned = [l for l in lines if l.strip() not in boilerplate and not page_marker.match(l.strip())]
    return "\n".join(cleaned)

In [89]:
# Chunking

def chunk_pdf(text):
    text = strip_boilerplate(text)
    return pdf_splitter.split_text(text)

def chunk_documents(text_content):
    all_chunks = []
    for doc in text_content:
        if doc["doc_type"] == "wikipedia":
            texts = chunk_article(doc["text"])
        elif doc["doc_type"] == "pdf":
            texts = chunk_pdf(doc["text"])
        else:
            continue

        for i, t in enumerate(texts):
            all_chunks.append({
                "text": t,
                "source": doc["source"],
                "section": doc["section"],
                "doc_type": doc["doc_type"],
                "chunk_id": f"{doc['source']}#{i}",
            })

    return all_chunks

all_chunks = chunk_documents(text_content)
print(f"Total Chunks: {len(all_chunks)}")

Total Chunks: 8219


In [ ]:
# Generating embeddings and upserting to Qdrant

if cloud_client.collection_exists(collection_name=collection_name):
    cloud_client.delete_collection(collection_name=collection_name)

cloud_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embed_dim,
        distance=models.Distance.COSINE)
)

def make_point_id(chunk_id):
    return hashlib.md5(chunk_id.encode()).hexdigest()

batch_size = 1000

for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i : i + batch_size]
    texts = [c["text"] for c in batch]

    result = voyageai_client.embed(texts, model=voyage_model, input_type="document")

    points = [
        models.PointStruct(
            id=make_point_id(c["chunk_id"]),
            vector=emb,
            payload={
                "text": c["text"],
                "source_url": c["source"],
                "section": c["section"],
                "doc_type": c["doc_type"],
            },
        )
        for c, emb in zip(batch, result.embeddings)
    ]

    cloud_client.upsert(
        collection_name=collection_name,
        points=points
    )
    print(f"batch {i // batch_size + 1}: embedded + upserted {len(points)} chunks")
    time.sleep(0.2)

In [6]:
class QdrantMotorsportsRetriever(BaseRetriever):
    top_k: int = 5
    doc_type_filter: Optional[str] = None

    def _get_relevant_documents(self, query: str) -> List[Document]:
        query_vec = voyageai_client.embed(
            [query], model=voyage_model, input_type="query"
        ).embeddings[0]

        query_filter = None
        if self.doc_type_filter:
            query_filter = models.Filter(
                must=[models.FieldCondition(
                    key="doc_type", match=models.MatchValue(value=self.doc_type_filter)
                )]
            )

        results = cloud_client.query_points(
            collection_name=collection_name,
            query=query_vec,
            query_filter=query_filter,
            limit=self.top_k,
        )
        return [
            Document(
                page_content=p.payload["text"],
                metadata={
                    "source_url": p.payload["source_url"],
                    "section": p.payload["section"],
                    "doc_type": p.payload["doc_type"],
                },
            )
            for p in results.points
        ]

retriever = QdrantMotorsportsRetriever(top_k=5)

In [7]:
SYSTEM_PROMPT = (
    "You are a Formula 1 regulations and history assistant. Answer using ONLY the " \
    "provided context. Cite sources inline using [1], [2], etc., matching the "
    "numbered context blocks. If the context doesn't contain the answer, say so "
    "explicitly rather than guessing."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\nQuestion: {question}"),
])

llm = ChatAnthropic(model=anthropic_model, max_tokens=1024)

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join(
        f"[{i+1}] Source: {d.metadata['source_url']} ({d.metadata['doc_type']})\n{d.page_content}"
        for i, d in enumerate(docs)
    )

def answer_question(question, top_k=5):
    retriever.top_k = top_k

    with langfuse.start_as_current_observation(
        name="answer_question",
        as_type="span",
        input={"question": question, "top_k": top_k},
    ) as span:
        docs = retriever.invoke(question,
                                config={"callbacks": [langfuse_handler]})
        context = format_docs(docs)

        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"context": context, "question": question},
                              config={"callbacks": [langfuse_handler]})

        sources = [
            {"n": i + 1, "url": d.metadata["source_url"], "doc_type": d.metadata["doc_type"]}
            for i, d in enumerate(docs)
        ]
        result = {"answer": answer, "sources": sources}

        span.update(output=result)
        return result

In [8]:
question = "Where is the Silverstone track located?"
result = answer_question(question)
print(f"Question: {question}")
print(f"Answer: {result['answer']}")
print(" === Sources ===")
for source in result["sources"]:
    print(f"[{source["n"]}] Source: {source["url"]} | Type: {source["doc_type"]}")

Question: Where is the Silverstone track located?
Answer: The Silverstone Circuit is a motor racing circuit in England, near the Northamptonshire villages of Silverstone and Whittlebury [2]. It straddles the Northamptonshire and Buckinghamshire border and is accessed from the nearby A43 [3]. The nearest city is Milton Keynes [3].
 === Sources ===
[1] Source: https://en.wikipedia.org/wiki/Silverstone_Circuit | Type: wikipedia
[2] Source: https://en.wikipedia.org/wiki/Silverstone_Circuit | Type: wikipedia
[3] Source: https://en.wikipedia.org/wiki/Silverstone_Circuit | Type: wikipedia
[4] Source: https://en.wikipedia.org/wiki/Silverstone_Circuit | Type: wikipedia
[5] Source: https://en.wikipedia.org/wiki/Silverstone_Circuit | Type: wikipedia


In [ ]:
questions = json.load(open("golden_set.json", encoding="utf-8"))